# Subject lists: one head-motion rule for every cohort

In [ ]:
import warnings
warnings.simplefilter("ignore", UserWarning)

import pandas as pd

from src.data import DataLoader, load_xcpd_motion
from src.config import DATASET_PARAMS

## Low-motion subject lists

In [ ]:
# ── Low-motion subject lists: one FD rule, applied per dataset ─────────────────
# Writes subject_ids_lowmotion.txt per dataset, keeping subjects with mean FD <= FD_THRESHOLD in every
# analysed task and session (re-export the registries afterwards), with HCP-YA excluded because it reports
# relative RMS, not FD, unless HCPYA_RMS_THRESHOLD sets an RMS cut (~0.11 is FD-equivalent).
FD_THRESHOLD = 0.2
HCPYA_RMS_THRESHOLD = None      # e.g. 0.11 to write an RMS-based list for HCP-YA

# Per dataset: the sessions that must pass (None pools all, a tuple restricts) and the FD source ('xcpd'
# per-run linc_qc or 'column' on df_behavior).
LOWMOTION_CONFIGS = [
    dict(dataset='WAND',        source='xcpd',   sessions=None),
    # Penn_LEAD must pass at session 1, the session every consumer analyses, but its list is written for
    # reference only because DATASET_PARAMS keeps it on the full subject list.
    dict(dataset='Penn_LEAD',   source='xcpd',   sessions=('1',)),
    dict(dataset='HCP-D',       source='column', column='meanFD', sessions=None),
]

lowmotion_rows = []
for cfg in LOWMOTION_CONFIGS:
    ds = cfg['dataset']
    loader = DataLoader(dataset=ds, load_behavior=(cfg['source'] == 'column'), verbose=False)
    tasks = DATASET_PARAMS[ds]['tasks']['MRI3T']

    if cfg['source'] == 'xcpd':
        long = load_xcpd_motion(loader)
        long = long[long['task'].isin(tasks)]
        # Label missing sessions explicitly, because groupby drops NaN keys and would silently empty the list.
        long = long.assign(session=long['session'].fillna(''))
        if cfg['sessions'] is not None:
            long = long[long['session'].isin(cfg['sessions'])]
        # One value per subject x task (x session, when sessions are pooled rather than selected).
        keys = ['subject_id', 'task'] if cfg['sessions'] is not None else ['subject_id', 'session', 'task']
        fd = long.groupby(keys)['mean_fd'].mean().unstack('task')
        complete = fd.notna().all(axis=1)
        under = (fd <= FD_THRESHOLD).all(axis=1)
        if cfg['sessions'] is None and 'session' in fd.index.names:
            # every session of a subject must pass, and every one must be complete
            complete = complete.groupby('subject_id').all()
            under = under.groupby('subject_id').all()
        keep = sorted(under.index[under & complete].astype(str))
        n_measured = int(complete.sum())
        quantity = 'FD (per-run, XCP-D)'
    else:
        col = loader.df_behavior[cfg['column']].dropna()
        keep = sorted(col.index[col <= FD_THRESHOLD].astype(str))
        n_measured = len(col)
        quantity = f'FD (per-subject, {cfg["column"]})'

    # The list on disk before the write below, which shows whether a re-run changed the cohort rather than
    # what the pre-FD cohort was.
    configured_file = DATASET_PARAMS[ds]['subject_ids_file']
    previous = loader.base_dir / configured_file
    n_on_disk = len([s for s in previous.read_text().split() if s.strip()]) if previous.exists() else None

    out_path = loader.base_dir / 'subject_ids_lowmotion.txt'
    with open(out_path, 'w') as f:
        f.write('\n'.join(f'sub-{s}' for s in keep) + '\n')

    lowmotion_rows.append(dict(dataset=ds, quantity=quantity, tasks=', '.join(tasks),
                               sessions='all' if cfg['sessions'] is None else ', '.join(cfg['sessions']),
                               n_measured=n_measured, n_retained=len(keep),
                               configured_file=configured_file, n_on_disk_before=n_on_disk,
                               path=str(out_path)))
    print(f'{ds:12s} {quantity:28s} measured {n_measured:5d} -> retained {len(keep):5d} '
          f'(on disk before this run: {n_on_disk}) -> {out_path.name}')

if HCPYA_RMS_THRESHOLD is not None:
    loader = DataLoader(dataset='HCP-YA', load_behavior=True, verbose=False)
    rms = loader.df_behavior[['meanRelRMS_rest', 'meanRelRMS_task']].dropna()
    keep = sorted(rms.index[(rms <= HCPYA_RMS_THRESHOLD).all(axis=1)].astype(str))
    out_path = loader.base_dir / 'subject_ids_lowmotion.txt'
    with open(out_path, 'w') as f:
        f.write('\n'.join(f'sub-{s}' for s in keep) + '\n')
    lowmotion_rows.append(dict(dataset='HCP-YA', quantity=f'relative RMS <= {HCPYA_RMS_THRESHOLD}',
                               tasks='rest, task (pooled)', sessions='all', n_measured=len(rms),
                               n_retained=len(keep),
                               configured_file=DATASET_PARAMS['HCP-YA']['subject_ids_file'],
                               n_on_disk_before=None, path=str(out_path)))
    print(f'HCP-YA       relative RMS <= {HCPYA_RMS_THRESHOLD}: retained {len(keep)} -> {out_path.name}')
else:
    print('HCP-YA       SKIPPED: reports relative RMS, not FD -- see the comment above.')

df_lowmotion = pd.DataFrame(lowmotion_rows)
df_lowmotion['used_by_config'] = df_lowmotion['configured_file'].eq('subject_ids_lowmotion.txt')
df_lowmotion[['dataset', 'quantity', 'sessions', 'n_measured', 'n_retained', 'n_on_disk_before',
              'configured_file', 'used_by_config']]